Preparacion

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalogo = "compras_rd"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.gold")

def silver(nombre):
    return spark.table(f"{catalogo}.silver.{nombre}")

def guardar(df, nombre):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{catalogo}.gold.{nombre}"))
    print(f"gold.{nombre} OK:", spark.table(f"{catalogo}.gold.{nombre}").count())

procesos = silver("procesos")

In [0]:
meses = F.array(*[F.lit(m) for m in [
    "enero","febrero","marzo","abril","mayo","junio",
    "julio","agosto","septiembre","octubre","noviembre","diciembre"]])

dim_fecha = (
    spark.sql("""SELECT explode(sequence(to_date('2023-01-01'),
                                         to_date('2027-12-31'),
                                         interval 1 day)) AS fecha""")
    .withColumn("fecha_key", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("nombre_mes", F.element_at(meses, F.col("mes")))
    .withColumn("dia_semana", F.dayofweek("fecha"))
    .select("fecha_key", "fecha", "anio", "trimestre", "mes", "nombre_mes", "dia_semana")
)

guardar(dim_fecha, "dim_fecha")

In [0]:
dim_institucion = (
    procesos.where("institucion_id IS NOT NULL")
    .groupBy("institucion_id")
    .agg(F.max("institucion").alias("institucion"))
)
guardar(dim_institucion, "dim_institucion")

w = Window.orderBy("modalidad", "modalidad_detalle")
dim_modalidad = (
    procesos.select("modalidad", "modalidad_detalle").distinct()
    .withColumn("modalidad_key", F.row_number().over(w))
)
guardar(dim_modalidad, "dim_modalidad")

In [0]:
dim_proveedor = (
    silver("proveedores")
    .select(
        F.col("party_id").alias("proveedor_id"),
        "rnc", "nombre", "escala", "genero", "region"
    )
)
guardar(dim_proveedor, "dim_proveedor")

display(dim_proveedor.groupBy("escala").count().orderBy(F.desc("count")))
display(dim_proveedor.groupBy("genero").count().orderBy(F.desc("count")))

In [0]:
p_base = (
    procesos.alias("p")
    .join(
        spark.table(f"{catalogo}.gold.dim_modalidad").alias("m"),
        F.col("p.modalidad").eqNullSafe(F.col("m.modalidad")) &
        F.col("p.modalidad_detalle").eqNullSafe(F.col("m.modalidad_detalle")),
        "left"
    )
    .select("p.*", "m.modalidad_key")
)

In [0]:
c = silver("contratos")
a = silver("adjudicaciones").select(
    "ocid", "award_id", "fecha_adjudicacion",
    "proveedor_id", "num_proveedores", "monto_adjudicado"
)

fact_contratos = (
    c.join(a, ["ocid", "award_id"], "left")
     .join(p_base.select("ocid", "institucion_id", "modalidad_key", "fecha_publicacion"),
           "ocid", "left")
     .withColumn("fecha_ref",
                 F.coalesce("fecha_adjudicacion", "inicio_contrato", "fecha_publicacion"))
     .withColumn("fecha_key", F.date_format("fecha_ref", "yyyyMMdd").cast("int"))
     .withColumn("duracion_dias", F.datediff("fin_contrato", "inicio_contrato"))
     .select(
         "ocid", "contrato_id", "award_id", "fecha_key",
         "institucion_id", "modalidad_key", "proveedor_id", "num_proveedores",
         "estado_contrato", "monto_contrato", "monto_adjudicado", "moneda",
         "inicio_contrato", "fin_contrato", "duracion_dias",
         "flag_monto_invalido", "flag_fecha_incoherente"
     )
)

guardar(fact_contratos, "fact_contratos")
print("Contratos en Silver:", c.count())

In [0]:
agg_c = (
    silver("contratos").groupBy("ocid")
    .agg(
        F.count("contrato_id").alias("num_contratos"),
        F.sum(F.when(~F.col("flag_monto_invalido"), F.col("monto_contrato")))
         .alias("monto_contratado")
    )
)

agg_a = (
    silver("adjudicaciones").groupBy("ocid")
    .agg(
        F.count("award_id").alias("num_adjudicaciones"),
        F.sum("monto_adjudicado").alias("monto_adjudicado")
    )
)

fact_procesos = (
    p_base.join(agg_c, "ocid", "left").join(agg_a, "ocid", "left")
    .withColumn("fecha_key",
                F.date_format(F.coalesce("inicio_oferta", "fecha_publicacion"),
                              "yyyyMMdd").cast("int"))
    .withColumn("dias_oferta", F.datediff("fin_oferta", "inicio_oferta"))
    .withColumn("dias_hasta_adjudicacion", F.datediff("fin_adjudicacion", "inicio_oferta"))
    .select(
        "ocid", "fecha_key", "institucion_id", "modalidad_key",
        "categoria", "estado", "moneda",
        "monto_estimado", "presupuesto", "num_oferentes",
        "num_adjudicaciones", "num_contratos",
        "monto_adjudicado", "monto_contratado",
        "inicio_oferta", "fin_oferta", "fin_adjudicacion",
        "dias_oferta", "dias_hasta_adjudicacion",
        "flag_monto_invalido"
    )
)

guardar(fact_procesos, "fact_procesos")

In [0]:
fc = spark.table(f"{catalogo}.gold.fact_contratos")
df_ = spark.table(f"{catalogo}.gold.dim_fecha")
di = spark.table(f"{catalogo}.gold.dim_institucion")
dp = spark.table(f"{catalogo}.gold.dim_proveedor")

validos = fc.where(~F.col("flag_monto_invalido"))

# Gasto mensual
agg_mensual = (
    validos.join(df_, "fecha_key", "left")
    .groupBy("anio", "mes", "moneda")
    .agg(F.count("*").alias("num_contratos"),
         F.sum("monto_contrato").alias("monto_total"),
         F.avg("monto_contrato").alias("monto_promedio"))
)
guardar(agg_mensual, "agg_gasto_mensual")

# Gasto por institución
agg_inst = (
    validos.join(di, "institucion_id", "left")
    .groupBy("institucion_id", "institucion", "moneda")
    .agg(F.count("*").alias("num_contratos"),
         F.sum("monto_contrato").alias("monto_total"))
)
guardar(agg_inst, "agg_gasto_institucion")

# Concentración de proveedores (Pareto)
tot = Window.partitionBy("moneda")
acum = (Window.partitionBy("moneda").orderBy(F.desc("monto_total"))
        .rowsBetween(Window.unboundedPreceding, Window.currentRow))

agg_prov = (
    validos.join(dp, "proveedor_id", "left")
    .groupBy("proveedor_id", "nombre", "escala", "genero", "moneda")
    .agg(F.count("*").alias("num_contratos"),
         F.sum("monto_contrato").alias("monto_total"))
    .withColumn("participacion", F.col("monto_total") / F.sum("monto_total").over(tot))
    .withColumn("participacion_acum", F.sum("participacion").over(acum))
    .withColumn("ranking", F.row_number().over(
        Window.partitionBy("moneda").orderBy(F.desc("monto_total"))))
)
guardar(agg_prov, "agg_proveedores")

In [0]:
for t in ["dim_fecha", "dim_institucion", "dim_modalidad", "dim_proveedor",
          "fact_contratos", "fact_procesos"]:
    print(t, spark.table(f"{catalogo}.gold.{t}").count())

# Cobertura de las uniones
display(spark.sql(f"""
SELECT count(*) AS contratos,
       sum(CASE WHEN fc.proveedor_id IS NULL THEN 1 ELSE 0 END) AS sin_proveedor,
       sum(CASE WHEN dp.proveedor_id IS NULL THEN 1 ELSE 0 END) AS proveedor_sin_dim,
       sum(CASE WHEN di.institucion_id IS NULL THEN 1 ELSE 0 END) AS sin_institucion,
       sum(CASE WHEN df.fecha_key IS NULL THEN 1 ELSE 0 END) AS fecha_fuera_de_dim
FROM {catalogo}.gold.fact_contratos fc
LEFT JOIN {catalogo}.gold.dim_proveedor dp ON fc.proveedor_id = dp.proveedor_id
LEFT JOIN {catalogo}.gold.dim_institucion di ON fc.institucion_id = di.institucion_id
LEFT JOIN {catalogo}.gold.dim_fecha df ON fc.fecha_key = df.fecha_key"""))

# Monedas
display(spark.sql(f"""
SELECT moneda, count(*) AS contratos, sum(monto_contrato) AS monto
FROM {catalogo}.gold.fact_contratos GROUP BY 1"""))

# Prueba rápida: las 10 instituciones con más gasto
display(spark.sql(f"""
SELECT institucion, moneda, num_contratos, round(monto_total, 0) AS monto_total
FROM {catalogo}.gold.agg_gasto_institucion
ORDER BY monto_total DESC LIMIT 10"""))